In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/em-dat-country-profiles/emdat-country-profiles_2025_01_20.xlsx
/kaggle/input/em-dat-country-profiles/metadata-emdat-country-profiles_2025_01_20-xlsx.csv
/kaggle/input/building-transforms/xview_geotransforms.json
/kaggle/input/dis-phase2-new-outputs/event_social_severity_urgency_phase2.json
/kaggle/input/dis-phase2-new-outputs/event_social_severity_urgency_phase2.csv
/kaggle/input/dis-phase0-outputs/aligned_data/aligned_index.csv
/kaggle/input/dis-phase0-outputs/aligned_data/disaster_universe.json
/kaggle/input/dis-phase0-outputs/aligned_data/time_alignment_plot.png
/kaggle/input/dis-phase0-outputs/aligned_data/emdat_country_priors.csv
/kaggle/input/dis-phase0-outputs/aligned_data/data_quality_report.json
/kaggle/input/dis-phase0-outputs/aligned_data/dataset_fingerprint.json
/kaggle/input/dis-phase0-outputs/aligned_data/geospatial_validation.json
/kaggle/input/dis-phase0-outputs/aligned_data/aligned_satellite.json
/kaggle/input/dis-phase0-outputs/aligned_data/missing_data_

In [2]:
import pandas as pd

fused_geo_df = pd.read_csv("/kaggle/input/dis-phase3-outputss/fused_risk_score_phase3.csv")
fused_geo_df.head()


,image_id,satellite_severity,num_buildings,event_id,social_severity,social_urgency,tweet_count,confidence,fused_risk,lat,lon
0,hurricane-matthew_00000295_post_disaster,0.000000,5,unknown_hurricane,0.464723,0.360558,17213,1.000000,0.211528,18.270571,-73.552850
1,socal-fire_00000723_post_disaster,0.277778,6,unknown_wildfire,0.474892,0.357765,4187,0.243246,0.352910,34.103173,-118.790296
2,midwest-flooding_00000033_post_disaster,0.000000,82,unknown_flood,0.403266,0.315843,9235,0.536513,0.184148,34.773097,-92.325008
3,socal-fire_00000960_post_disaster,0.000000,40,unknown_wildfire,0.474892,0.357765,4187,0.243246,0.214021,34.062402,-118.745582
4,midwest-flooding_00000348_post_disaster,0.000000,5,unknown_flood,0.403266,0.315843,9235,0.536513,0.184148,36.201939,-96.222788


In [3]:
K = 30


In [4]:
from sklearn.cluster import KMeans
import numpy as np

coords = fused_geo_df[["lat", "lon"]].values


In [5]:
kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

fused_geo_df["zone_id"] = kmeans.fit_predict(coords)


In [6]:
zone_centroids = (
    fused_geo_df
    .groupby("zone_id")[["lat", "lon"]]
    .mean()
    .reset_index()
    .rename(columns={"lat": "zone_lat", "lon": "zone_lon"})
)


In [7]:
zone_risk = (
    fused_geo_df
    .groupby("zone_id")
    .agg(
        zone_mean_risk=("fused_risk", "mean"),
        zone_max_risk=("fused_risk", "max"),
        point_count=("fused_risk", "count")
    )
    .reset_index()
)


In [8]:
zone_df = zone_centroids.merge(
    zone_risk,
    on="zone_id",
    how="inner"
)

zone_df.head()


,zone_id,zone_lat,zone_lon,zone_mean_risk,zone_max_risk,point_count
0,0,34.092442,-118.884523,0.405755,0.714021,135
1,1,-0.883908,119.864415,0.309610,0.697311,79
2,2,30.167009,-85.643764,0.347312,0.711528,169
3,3,19.309022,-99.186911,0.199523,0.206402,120
4,4,18.261032,-74.102223,0.676573,0.705118,12


In [9]:
zone_df = zone_df.sort_values(
    by="zone_mean_risk",
    ascending=False
).reset_index(drop=True)


In [10]:
fused_geo_df.to_csv(
    "/kaggle/working/fused_risk_with_zones_phase4.csv",
    index=False
)


In [11]:
zone_df.to_csv(
    "/kaggle/working/zone_summary_phase4.csv",
    index=False
)
